# Proyecto Final: Agente Experto en Ganancia Muscular

Este cuaderno contiene la solución del proyecto final adaptada al dominio de **Entrenamiento, Nutrición y Descanso** para hipertrofia muscular.

In [8]:
import os
import glob
import shutil
import operator
from typing import Annotated, TypedDict
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("No se encontró la API Key.")
else:
    print("API Key cargada.")

API Key cargada.


In [9]:
PERSIST_DIR = "./chroma_db_fitness"

archivos_pdf = [f for f in glob.glob("*.pdf") if not os.path.basename(f).startswith("Proyecto Final")]

documentos = []
for archivo in archivos_pdf:
    loader = PyPDFLoader(archivo)
    documentos.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
fragmentos = text_splitter.split_documents(documentos)
print(f"Textos divididos en {len(fragmentos)} fragmentos (chunks).")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2", 
    google_api_key=API_KEY
)

# Se conecta a la base existente y borra el contenido interno para evitar duplicación
vector_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings
)

try:
    vector_store.delete_collection()
    print("Hay datos anteriores, eliminándose")
except Exception:
    pass

# Se crea la nueva colección con los fragmentos limpios
vector_store = Chroma.from_documents(
    documents=fragmentos, 
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})
print("Base de datos cargada.")

Textos divididos en 30 fragmentos (chunks).
Hay datos anteriores, eliminándose
Base de datos cargada.


In [10]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.0, google_api_key=API_KEY)

contextualize_q_system_prompt = """Dada la historia del chat y la última pregunta del usuario \
(que podría hacer referencia al historial), formula una pregunta independiente que se entienda \
sin el historial. Si la pregunta ya es independiente, devuélvela exactamente igual. \
NO la respondas, solo reformúlala. \nUsa términos técnicos explícitos y desambigua: si el usuario habla de "descansar" por la noche, \nescribe "sueño" o "privación de sueño"; si habla de descanso entre series, dilo así."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
reformular_cadena = contextualize_q_prompt | llm | StrOutputParser()

# 2. Interceptor de búsqueda en ChromaDB
def obtener_contexto(x):
    if x.get("chat_history"):
        pregunta_busqueda = reformular_cadena.invoke(x).strip()
        print(f"Búsqueda en BD vectorizada: {pregunta_busqueda}")
    else:
        pregunta_busqueda = x["input"]
        
    docs = retriever.invoke(pregunta_busqueda)
    print(f"Fuentes recuperadas: {[os.path.basename(d.metadata.get('source', '?')) for d in docs]}")
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Prompt Principal
system_prompt = """
Eres un entrenador y consultor experto en ciencias del deporte, hipertrofia muscular, nutrición deportiva y descanso.

REGLAS DE COMPORTAMIENTO:
1. Utiliza ÚNICAMENTE la información proporcionada en el siguiente <contexto>. 
2. Si el contexto solo cubre parte de la pregunta, responde con lo que sí contiene e indica qué parte no está documentada. Solo si no hay NADA relacionado en el contexto, responde explícitamente: "No dispongo de información suficiente en mis documentos sobre ese tema."
3. Responde de forma clara, directa y estructurada.

<contexto>
{context}
</contexto>
"""
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

rag_chain = (
    {
        "context": obtener_contexto, 
        "input": lambda x: x["input"], 
        "chat_history": lambda x: x.get("chat_history", [])
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# 4. Estado de LangGraph
class AgentState(TypedDict):
    chat_history: Annotated[list[BaseMessage], operator.add]
    input: str
    answer: str

def procesar_consulta(state: AgentState):
    respuesta = rag_chain.invoke({
        "input": state["input"],
        "chat_history": state["chat_history"]
    })
    
    return {
        "answer": respuesta,
        "chat_history": [
            HumanMessage(content=state["input"]), 
            AIMessage(content=respuesta)
        ]
    }

flujo = StateGraph(AgentState)
flujo.add_node("experto", procesar_consulta)
flujo.add_edge(START, "experto")
flujo.add_edge("experto", END)

# 5. Integración nativa de memoria
checkpointer = MemorySaver()
agente_entrenador = flujo.compile(checkpointer=checkpointer)

In [11]:
# Configuración del thread de LangGraph
config = {"configurable": {"thread_id": "sesion-entrenamiento-1"}}

def interactuar_con_agente(pregunta: str):
    print(f"\nUsuario: {pregunta}")
    
    # Se envía solo el input; LangGraph inyecta y guarda el chat_history automáticamente
    resultado = agente_entrenador.invoke({"input": pregunta}, config=config)
    
    print(f"Asistente Experto: {resultado['answer']}")
    print("-" * 60)

print("--- INICIANDO PRUEBAS DOCUMENTADAS DEL PROYECTO ---")

interactuar_con_agente("¿Cuáles son los principios del entrenamiento para ganar masa muscular?")
interactuar_con_agente("Según los documentos, ¿qué tipo de proteína o suplementación es la más recomendable para la síntesis proteica?")
interactuar_con_agente("Teniendo en cuenta esa proteína que mencionas, ¿hay algún momento estratégico del día para tomarla?")
interactuar_con_agente("¿Qué ocurre si no descanso durante 1 o 2 noches? ¿Perderé fuerza?")
interactuar_con_agente("¿Cuál es la capital de Australia?")

--- INICIANDO PRUEBAS DOCUMENTADAS DEL PROYECTO ---

Usuario: ¿Cuáles son los principios del entrenamiento para ganar masa muscular?
Fuentes recuperadas: ['Nutrición.pdf', 'Nutrición.pdf', 'Nutrición.pdf', 'Nutrición.pdf', 'Nutrición.pdf']


c:\Users\Usuario\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Asistente Experto: Basándome en el documento proporcionado, los principios y pautas clave para estructurar el entrenamiento enfocado en ganar masa muscular (hipertrofia) son los siguientes:

*   **Frecuencia Semanal:** Entrenar cada grupo muscular 3 días a la semana (por ejemplo, mediante rutinas de cuerpo completo o *full-body*) es la frecuencia más adecuada y suficiente para optimizar resultados.
*   **Volumen de Entrenamiento:** Se recomienda realizar entre 3 y 5 series por ejercicio, aumentando este volumen a medida que el deportista gana experiencia. Existe una relación directa entre el volumen semanal y la hipertrofia.
*   **Intensidad y Repeticiones:** Se deben realizar series de 6 a 12 repeticiones. No es estrictamente necesario llegar al fallo muscular absoluto; entrenar dejando entre 0 y 2 repeticiones en reserva (cerca del fallo) produce el mismo crecimiento evitando la fatiga excesiva y el sobreentrenamiento.
*   **Descanso entre series:** Descansar entre 2 y 3 minutos es m

c:\Users\Usuario\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Búsqueda en BD vectorizada: ¿Qué tipo de proteína o suplementación es la más recomendable para la síntesis proteica según los documentos?
Fuentes recuperadas: ['Nutrición.pdf', 'Nutrición.pdf', 'Nutrición.pdf', 'Nutrición.pdf', 'Nutrición.pdf']


c:\Users\Usuario\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


... (9 outputs left out)

c:\Users\Usuario\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Búsqueda en BD vectorizada: ¿Cuál es la capital de Australia?
Fuentes recuperadas: ['Hipertrofia.pdf', 'Descanso Nocturno.pdf', 'Descanso entre Series.pdf', 'Descanso Nocturno.pdf', 'RIR.pdf']


c:\Users\Usuario\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Asistente Experto: No dispongo de información suficiente en mis documentos sobre ese tema.
------------------------------------------------------------
